# TwoTower Model

Neural two-tower recommender, ranking task, compared against SVD via graded NDCG.

In [ ]:
# Imports
import numpy as np
import pandas as pd

from libreco.data import DatasetPure
from libreco.algorithms import TwoTower

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [ ]:
# Load Data
train = pd.read_csv("train_ratings.csv")
val   = pd.read_csv("val_ratings.csv")
test  = pd.read_csv("test_ratings.csv")

print(f"Train: {len(train):,}  |  Val: {len(val):,}  |  Test: {len(test):,}")
print("Columns:", list(train.columns))

## 1. Data Preparation

- `LibRecommender` requires columns named `user`, `item`, `label` (user/item must be the first two columns). 
- We rename our columns and set `label = 1` for all interactions — the "all interactions = positive" decision for ranking.

In [ ]:
def to_ranking_format(df):
    """Rename to LibRecommender's expected columns; all interactions = positive."""
    out = df.rename(columns={"user_id": "user", "book_id": "item"}).copy()
    out["label"] = 1
    return out[["user", "item", "label"]]   # user/item must be first two columns

train_rk = to_ranking_format(train)
val_rk   = to_ranking_format(val)

print(train_rk.head())
print("\nShape:", train_rk.shape)
print("All labels = 1:", (train_rk['label'] == 1).all())

## 2. Build Dataset and Initialize Model

Build the `LibRecommender` dataset objects, then instantiate TwoTower with `task="ranking"` (the only task it supports). 

In [ ]:
# Build LibRecommender dataset objects.
# build_trainset returns (transformed data, data_info); data_info holds
# user/item counts and mappings used throughout training and prediction.
# build_evalset prepares the validation data for per-epoch monitoring.

train_data, data_info = DatasetPure.build_trainset(train_rk)
eval_data = DatasetPure.build_evalset(val_rk)

print(data_info)

In [ ]:
# Reset graph (TF1 mode — avoids "Variable already exists" on re-run)
import tensorflow as tf
tf.compat.v1.reset_default_graph()

# FINAL TwoTower model — config locked after manual hyperparameter search.
# Full search recorded in notes_twotower_implementation.md.
twotower = TwoTower(
    task="ranking",                # only task TwoTower supports
    data_info=data_info,
    loss_type="softmax",           # in-batch negatives (Yi et al. 2019); fits top-K retrieval
    embed_size=32,                 # 32 > 64 (64 unstable on sparse data)
    norm_embed=True,               # L2-normalise tower outputs → score is cosine in [-1,1]
    n_epochs=12,                   # early-stopping point: val ndcg plateaus ~ep12 (0.408),
                                   #   train_loss keeps falling after → mild overfit beyond 12
    lr=0.001,                      # 0.001 > 0.003
    batch_size=2048,               # governs # of in-batch negatives under softmax
    num_neg=1,                     # inert under softmax (negatives are in-batch); kept default
    use_bn=True,
    hidden_units=(128, 64, 32),    # architecture of each tower
    temperature=0.1,               # ⭐ decisive. Swept 1.0 / 0.15 / 0.1 / 0.05 → inverted-U,
                                   #   peak at 0.1: ndcg 0.085 / 0.373 / 0.394 / 0.394.
                                   #   Low temp required: norm_embed compresses scores to [-1,1].
    seed=RANDOM_STATE,
)
print("Final TwoTower initialised (n_epochs=12)")

## 3. Model Training

Final training run with the locked configuration (see hyperparameter search in notes). 
- `n_epochs=12` chosen as the early-stopping point where validation NDCG plateaus. 
- `eval_user_num=None` evaluates on the full validation set for an accurate per-epoch monitoring curve.

In [ ]:
# Final training run (n_epochs=12, locked config).
twotower.fit(
    train_data,
    neg_sampling=True,    # required for ranking with positive-only data
    verbose=2,
    shuffle=True,         # VAL set, monitoring only
    eval_data=eval_data,
    metrics=["loss", "precision", "recall", "ndcg"],
    k=10,
    eval_user_num=None,  # full val set (~4-5 min/epoch).
)

## 4. Evaluation Setup

### 4.1 Data Structures

Prepare the per-user data structures used for graded-NDCG evaluation:
each user's train items (to exclude when sampling negatives), their held-out
test ratings (graded relevance source), and the full item list.

In [ ]:
# Books each user interacted with in TRAIN (to exclude when sampling negatives)
train_items_by_user = train.groupby("user_id")["book_id"].apply(set).to_dict()

# Held-out TEST ratings per user: {user: {book: rating}} — graded relevance source
test_ratings_by_user = (
    test.groupby("user_id")
        .apply(lambda g: dict(zip(g["book_id"], g["rating"])))
        .to_dict()
)

all_items = train["book_id"].unique()

print(f"Users with test data: {len(test_ratings_by_user):,}")
print(f"Total unique items:   {len(all_items):,}")
# sanity check one user
u0 = next(iter(test_ratings_by_user))
print(f"Example user {u0[:8]}... has {len(test_ratings_by_user[u0])} test ratings")

### 4.2 Shared Candidate Pools

For graded-NDCG comparison, every model is scored on the SAME candidate pool per
user: their held-out test items (graded by true rating) + 100 sampled negatives
drawn from non-interacted items. Built once with a fixed seed so Popularity /
TwoTower / SVD are evaluated on identical candidates — a controlled comparison.
Following the sampled-evaluation protocol of He et al. (2017).

In [ ]:
def build_candidate_pools(test_ratings_by_user, train_items_by_user,
                          all_items, n_neg=100, seed=RANDOM_STATE):
    """For each test user: pool = their test items + n_neg sampled negatives.
    Negatives exclude anything the user interacted with in train.
    Built once and reused across all models for a fair comparison."""
    rng = np.random.default_rng(seed)
    all_items_arr = np.asarray(all_items)
    pools = {}

    for user, test_items in test_ratings_by_user.items():
        seen = train_items_by_user.get(user, set())          # exclude train items
        pos_items = set(test_items.keys())                   # held-out test items (graded)
        exclude = seen | pos_items

        # sample negatives not in exclude (oversample then filter to be safe)
        negs = []
        while len(negs) < n_neg:
            cand = rng.choice(all_items_arr, size=n_neg * 2, replace=False)
            negs = [it for it in cand if it not in exclude][:n_neg]

        pools[user] = list(pos_items) + negs                 # full candidate pool

    return pools

# Build once
candidate_pools = build_candidate_pools(
    test_ratings_by_user, train_items_by_user, all_items, n_neg=100
)
print(f"Built candidate pools for {len(candidate_pools):,} users")
u0 = next(iter(candidate_pools))
print(f"Example user pool size: {len(candidate_pools[u0])} "
      f"({len(test_ratings_by_user[u0])} test + {len(candidate_pools[u0]) - len(test_ratings_by_user[u0])} neg)")

### 4.3 Graded NDCG Evaluation

A single evaluation function shared by all models. For each user, the model
scores their candidate pool; we compute graded NDCG@K using the true test
ratings as relevance (via sklearn). Only the `score_fn` differs between models —
the candidate pools, ground truth, and metric code are identical.

In [ ]:
from sklearn.metrics import ndcg_score

def evaluate_ndcg(score_fn, candidate_pools, test_ratings_by_user, k=10):
    """Mean graded NDCG@k over all users.
    score_fn(user, items) -> list of predicted scores (one per item, ranking only).
    Relevance = true test rating (0 if the item isn't a held-out positive)."""
    ndcgs = []

    for user, items in candidate_pools.items():
        ratings = test_ratings_by_user[user]                 # {book: true rating}
        y_true = [ratings.get(it, 0) for it in items]        # graded relevance (0 = negative)

        # need at least one positive and one non-positive to rank
        if sum(y_true) == 0 or len(set(y_true)) == 1:
            continue

        y_score = score_fn(user, items)                      # model's predicted scores
        ndcgs.append(ndcg_score([y_true], [y_score], k=k))

    return float(np.mean(ndcgs)), len(ndcgs)

In [ ]:
# How many users does evaluate_ndcg actually score vs skip?
total_users = len(candidate_pools)
scored = 0
skipped_no_pos = 0
skipped_uniform = 0
for user, items in candidate_pools.items():
    ratings = test_ratings_by_user[user]
    y_true = [ratings.get(it, 0) for it in items]
    if sum(y_true) == 0:
        skipped_no_pos += 1
    elif len(set(y_true)) == 1:
        skipped_uniform += 1
    else:
        scored += 1
print(f"Total users:        {total_users:,}")
print(f"Scored:             {scored:,} ({scored/total_users*100:.1f}%)")
print(f"Skipped (no pos):   {skipped_no_pos:,}")
print(f"Skipped (uniform):  {skipped_uniform:,}")

In [ ]:
def make_twotower_score_fn(model):
    """Robust score_fn for any TwoTower model. Auto-casts candidate ids to the
    model's data_info item-id key type (DatasetPure=int64, DatasetFeat=str),
    preventing the silent 'unknown interaction' corruption from id-type mismatch."""
    sample_key = list(model.data_info.item2id.keys())[0]
    id_cast = type(sample_key)
    is_str = isinstance(sample_key, str)

    def score_fn(user, items):
        items_cast = [id_cast(it) for it in items]
        users = [str(user)] * len(items_cast) if is_str else [user] * len(items_cast)
        return model.predict(user=users, item=items_cast, cold_start="popular")
    return score_fn


def sanity_check_score_fn(score_fn, candidate_pools, n_check=20):
    """Verify score_fn returns non-default scores for known items before trusting eval.
    If most candidates score the default (0.0), id types likely mismatch -> abort."""
    sample_users = list(candidate_pools.keys())[:n_check]
    n_default = total = 0
    for u in sample_users:
        scores = np.asarray(score_fn(u, candidate_pools[u]), dtype=float)
        n_default += int((scores == 0.0).sum())
        total += len(scores)
    frac = n_default / total
    ok = frac < 0.5
    print(f"Sanity check: {frac:.1%} default scores → {'OK' if ok else 'FAIL — id mismatch!'}")
    return ok

## 5. Popularity Baseline

A non-personalized lower bound: score every book by how many users rated it in
the training set (popularity), ignoring who the user is. Any personalized model
that fails to beat this hasn't justified its complexity. This is also the
simplest model, so it validates the evaluation pipeline end-to-end.

In [ ]:
# Book popularity = number of training interactions per book.
book_popularity = train["book_id"].value_counts().to_dict()

def popularity_score_fn(user, items):
    """Score = book's training popularity. Same for every user (non-personalized)."""
    return [book_popularity.get(it, 0) for it in items]

pop_ndcg, n_users = evaluate_ndcg(
    popularity_score_fn, candidate_pools, test_ratings_by_user, k=10
)
print(f"Popularity baseline — graded NDCG@10: {pop_ndcg:.4f}  (over {n_users:,} users)")

### Diagnostic — Popularity bias in negative sampling

Positives (user-read books) are far more popular than randomly sampled negatives,
which is why the popularity baseline scores artificially high. Evidence below.

In [ ]:
# Evidence of popularity bias: positives (books users actually read) skew popular,
# while random negatives are mostly long-tail. This explains the inflated
# popularity-baseline NDCG. Sampled over 5,000 users.
import numpy as np

pos_pops, neg_pops = [], []
for user, items in list(candidate_pools.items())[:5000]:   # sample 5,000 users
    ratings = test_ratings_by_user[user]
    for it in items:
        pop = book_popularity.get(it, 0)
        if it in ratings:
            pos_pops.append(pop)
        else:
            neg_pops.append(pop)

print(f"Positives — mean popularity: {np.mean(pos_pops):.1f}  (median {np.median(pos_pops):.0f})")
print(f"Negatives — mean popularity: {np.mean(neg_pops):.1f}  (median {np.median(neg_pops):.0f})")
print(f"Positive/negative popularity ratio: {np.mean(pos_pops)/max(np.mean(neg_pops),1):.1f}x")

## 6. TwoTower — Graded NDCG

Score each user's candidate pool with the trained TwoTower model, on the SAME
pools as the popularity baseline. This is the decisive comparison: can TwoTower
beat the popularity baseline (0.6992), i.e. learn signal beyond popularity?

In [ ]:
# Robust score_fn (auto-casts ids to the model's id type) + sanity check before eval.
twotower_score_fn = make_twotower_score_fn(twotower)
assert sanity_check_score_fn(twotower_score_fn, candidate_pools), "ID-only failed sanity check"

tt_ndcg, n_users = evaluate_ndcg(
    twotower_score_fn, candidate_pools, test_ratings_by_user, k=10
)
print(f"TwoTower — graded NDCG@10: {tt_ndcg:.4f}  (over {n_users:,} users)")
print(f"Popularity baseline:        0.6992")
print(f"Difference:                 {tt_ndcg - 0.6992:+.4f}")

# Side Features (Stretch Experiment)

Can adding item-side content features (book attributes) to the TwoTower item tower
improve ranking beyond the pure-ID model?

**Method.** Features come from `book_features.csv` (extracted in EDA). Dense features are
Min-Max scaled to [0,1] — LibRecommender encodes them multiplicatively (`embed_var * value`),
which needs bounded, non-negative inputs. The model is otherwise identical to the locked
ID-only config, so any difference isolates the feature's effect.

**Key finding.** Concatenated dense features dilute the item-ID embedding: the item-tower
input grows to `embed_size × (1 + n_features)`, so the ID share is `1/(1+n_features)`. Three
features collapsed (NDCG ~0.12); a single informative feature works. `average_rating` is used
here — `num_pages` performed comparably, while `publication_year` collapsed even alone (a weak,
noisy signal). Full ablation in `notes_twotower_implementation.md`.

**Result.** A single `average_rating` feature reaches graded NDCG@10 **0.8550** vs ID-only
**0.8508** (+0.0042) — a tiny but real gain. A well-trained ID embedding already captures most
of this global-quality signal, so the marginal value of simple content features is small;
richer features (e.g. review-text embeddings) are left as future work.

In [ ]:
# Load CLEAN raw features (never overwrite the CSV with scaled values)
book_feat = pd.read_csv("book_features.csv")
content_cols = ["average_rating", "num_pages", "publication_year"]
book_feat = book_feat[["book_id"] + content_cols]
book_feat["book_id"] = book_feat["book_id"].astype(str)

# LibRecommender encodes dense features multiplicatively (embed_var * value),
# which needs non-negative, bounded inputs. Clip outliers (1-99 pct) then Min-Max
# to [0,1]. (mean0/std1 fails here: negatives/outliers blow up the multiplied embeds.)
for col in content_cols:
    lo, hi = book_feat[col].quantile([0.01, 0.99])
    book_feat[col] = book_feat[col].clip(lo, hi)
    rng = book_feat[col].max() - book_feat[col].min()
    book_feat[col] = (book_feat[col] - book_feat[col].min()) / rng

print("Scaled to [0,1]:")
print(book_feat[content_cols].describe().loc[["min", "max", "mean"]])

def add_features(df_rk_with_item):
    out = df_rk_with_item.copy()
    out["item"] = out["item"].astype(str)
    out = out.merge(book_feat, left_on="item", right_on="book_id", how="left")
    return out.drop(columns=["book_id"])

train_feat = add_features(train_rk)
val_feat   = add_features(val_rk)
print("\nShape:", train_feat.shape, "| missing:", train_feat[content_cols].isna().any().any())

In [ ]:
# Confirm the content features are attached to data_info for the feature model.
# dense_col should now list the 3 content features (vs empty for the ID-only DatasetPure).

print("n_users:", data_info.n_users, "n_items:", data_info.n_items)
print("sparse_col:", getattr(data_info, 'sparse_col', None))
print("dense_col:", getattr(data_info, 'dense_col', None))

In [ ]:
from libreco.data import DatasetFeat
import tensorflow as tf
tf.compat.v1.reset_default_graph()

# FINAL feature model: single content feature average_rating (quality signal).
# One feature keeps item-id share at 1/2 (no dilution); average_rating is genuinely
# informative (unlike publication_year which collapsed). Same locked config as ID-only
# otherwise, to isolate the feature's effect.
one_feat = ["average_rating"]
train_data, data_info = DatasetFeat.build_trainset(
    train_feat, user_col=[], item_col=one_feat, sparse_col=[], dense_col=one_feat,
)
eval_data = DatasetFeat.build_evalset(val_feat)

twotower_feat = TwoTower(
    task="ranking", data_info=data_info, loss_type="softmax",
    embed_size=32, norm_embed=True, n_epochs=12, lr=0.001, batch_size=2048,
    num_neg=1, use_bn=True, hidden_units=(128, 64, 32), temperature=0.1, seed=RANDOM_STATE,
)
twotower_feat.fit(train_data, neg_sampling=True, verbose=2, shuffle=True,
                  eval_data=eval_data, metrics=["loss","precision","recall","ndcg"],
                  k=10, eval_user_num=2000)

In [ ]:
# Same robust pattern — make_twotower_score_fn auto-detects this model's str id type.
twotower_feat_score_fn = make_twotower_score_fn(twotower_feat)
assert sanity_check_score_fn(twotower_feat_score_fn, candidate_pools), "Feature failed sanity check"

feat_ndcg, n_users = evaluate_ndcg(
    twotower_feat_score_fn, candidate_pools, test_ratings_by_user, k=10
)
print(f"TwoTower + average_rating — graded NDCG@10: {feat_ndcg:.4f}  (over {n_users:,} users)")
print(f"TwoTower ID-only:           {tt_ndcg:.4f}")
print(f"Difference:                 {feat_ndcg - tt_ndcg:+.4f}")